## 1. Configuration


In [ ]:
import os, glob, time, copy, json, random, warnings
warnings.filterwarnings("ignore")

import cv2
cv2.setNumThreads(0)          # OpenCV threads + forked DataLoader workers deadlock on Colab
import numpy as np
import pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, balanced_accuracy_score,
                             confusion_matrix, classification_report)
import matplotlib.pyplot as plt
import seaborn as sns

SEED = 42     # same seed as Lab 02 -> identical subset + split, so Set A is directly comparable
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BEST_MODELS = ["ResNet50", "DenseNet121", "ResNet101"]   # Lab 01's top three, for reference
CNN_MODELS  = ["ResNet50", "DenseNet121"]                 # CNN Model 1 / CNN Model 2 for Task 4-5
BEST_FILTER = "Gaussian"                                   # Set B, see note above

MAX_PER_CLASS = 500      # same cap as Lab 02 -> same subset
IMG_SIZE      = 224
BATCH_SIZE    = 32
EPOCHS        = 10
LR            = 1e-4
NUM_WORKERS   = 2

ON_KAGGLE = os.path.isdir("/kaggle/input")
OUT = "/kaggle/working/lab03" if ON_KAGGLE else "/content/lab03"
for d in ["figures", "confusion", "curves", "reports"]:
    os.makedirs(f"{OUT}/{d}", exist_ok=True)

print(f"device={DEVICE}  CNN_MODELS={CNN_MODELS}  best_filter={BEST_FILTER}  epochs={EPOCHS}")


## 2. Dataset preparation identical to Lab 02


In [ ]:
# locate HAM10000: Kaggle input if attached, otherwise download via kagglehub (Colab: token in Secrets)
hits = glob.glob("/kaggle/input/**/HAM10000_metadata.csv", recursive=True) if ON_KAGGLE else []
if not hits:
    try:
        from google.colab import userdata
        os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
    except Exception:
        pass
    import kagglehub
    base = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
    hits = glob.glob(f"{base}/**/HAM10000_metadata.csv", recursive=True)
assert hits, "HAM10000 not found. On Colab add a KAGGLE_API_TOKEN secret; on Kaggle attach the dataset."
meta_path = hits[0]
root = os.path.dirname(meta_path)
img_paths = {os.path.splitext(os.path.basename(p))[0]: p
             for p in glob.glob(f"{root}/**/*.jpg", recursive=True)}

meta = pd.read_csv(meta_path)
meta["path"] = meta["image_id"].map(img_paths)
meta = meta.dropna(subset=["path"]).reset_index(drop=True)

CLASS_NAMES = {
    "akiec": "Actinic keratoses", "bcc": "Basal cell carcinoma", "bkl": "Benign keratosis",
    "df": "Dermatofibroma", "mel": "Melanoma", "nv": "Melanocytic nevi", "vasc": "Vascular lesions",
}
CLASSES = sorted(meta["dx"].unique())
NUM_CLASSES = len(CLASSES)
cls2idx = {c: i for i, c in enumerate(CLASSES)}

print("Full HAM10000:", len(meta), "images")
print(meta["dx"].value_counts().to_string())


In [ ]:
# subset: cap each class at MAX_PER_CLASS, sampled by lesion so duplicates of a lesion stay together
def cap_by_lesion(df, cap, seed=SEED):
    parts = []
    for cls, g in df.groupby("dx"):
        if len(g) <= cap:
            parts.append(g); continue
        lesions = g["lesion_id"].drop_duplicates().sample(frac=1, random_state=seed)
        keep, n = [], 0
        for lid in lesions:
            rows = g[g["lesion_id"] == lid]
            if n + len(rows) > cap: continue
            keep.append(rows); n += len(rows)
        parts.append(pd.concat(keep))
    return pd.concat(parts).sample(frac=1, random_state=seed).reset_index(drop=True)

sub = cap_by_lesion(meta, MAX_PER_CLASS)
sub["label"] = sub["dx"].map(cls2idx)

# group-aware stratified split: images of the same lesion never cross train/val/test
y, groups = sub["label"].values, sub["lesion_id"].values
outer = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED)
trval_idx, test_idx = next(outer.split(sub, y, groups))
inner = StratifiedGroupKFold(n_splits=6, shuffle=True, random_state=SEED)
tr_rel, va_rel = next(inner.split(sub.iloc[trval_idx], y[trval_idx], groups[trval_idx]))
train_idx, val_idx = trval_idx[tr_rel], trval_idx[va_rel]

splits = {"train": sub.iloc[train_idx], "val": sub.iloc[val_idx], "test": sub.iloc[test_idx]}
dist = pd.DataFrame({k: v["dx"].value_counts() for k, v in splits.items()}).fillna(0).astype(int)
dist["total"] = dist.sum(1)
print(f"Subset: {len(sub)} images  |  train {len(train_idx)}  val {len(val_idx)}  test {len(test_idx)}\n")
print(dist.to_string())

counts = np.bincount(splits["train"]["label"], minlength=NUM_CLASSES)
class_weights = torch.tensor(counts.sum() / (NUM_CLASSES * counts), dtype=torch.float32).to(DEVICE)

# one representative image per class, used for Tasks 1-3 (the brief asks for "representative
# images", not the full dataset, for the qualitative/edge-detector comparisons)
sample_rows = sub.groupby("dx").first().reset_index()
sample_imgs = {}
for _, row in sample_rows.iterrows():
    img = cv2.cvtColor(cv2.imread(row["path"]), cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    sample_imgs[row["dx"]] = img
print("\nSample images for Tasks 1-3:", list(sample_imgs.keys()))


## 3. Task 1 -- Comparative Edge Detection


In [ ]:
PREWITT_KX = np.array([[1, 0, -1], [1, 0, -1], [1, 0, -1]], dtype=np.float32)
PREWITT_KY = np.array([[1, 1, 1], [0, 0, 0], [-1, -1, -1]], dtype=np.float32)


def to_gray(img_rgb):
    return cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

def sobel_gx(gray): return cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
def sobel_gy(gray): return cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)

def sobel_magnitude(gray):
    return cv2.magnitude(sobel_gx(gray), sobel_gy(gray))

def prewitt(gray):
    px = cv2.filter2D(gray.astype(np.float32), -1, PREWITT_KX)
    py = cv2.filter2D(gray.astype(np.float32), -1, PREWITT_KY)
    return cv2.magnitude(px, py)

def laplacian(gray):
    return cv2.Laplacian(gray, cv2.CV_64F, ksize=3)

def log_edge(gray, k=5, sigma=1.0):
    blurred = cv2.GaussianBlur(gray, (k, k), sigma)
    return cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)

def canny_edge(gray, low=30, high=100, k=3):
    if k > 1:
        gray = cv2.GaussianBlur(gray, (k, k), 0)
    return cv2.Canny(gray, low, high)

def to_display(edge_resp):
    '''Any float/uint8 edge response -> normalized uint8 for imshow.'''
    mag = np.abs(edge_resp.astype(np.float64))
    return cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)

def to_binary(edge_resp):
    disp = to_display(edge_resp)
    _, binary = cv2.threshold(disp, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return binary

DETECTOR_ORDER = ["Sobel", "Prewitt", "Laplacian", "LoG", "Canny"]

def run_detectors(gray, canny_low=30, canny_high=100, canny_k=3):
    return {
        "Sobel":     sobel_magnitude(gray),
        "Prewitt":   prewitt(gray),
        "Laplacian": laplacian(gray),
        "LoG":       log_edge(gray),
        "Canny":     canny_edge(gray, canny_low, canny_high, canny_k),
    }


In [ ]:
# Required figure: Original -> Sobel -> Prewitt -> Laplacian -> LoG -> Canny, >= 3 classes
FIGURE_CLASSES = ["nv", "mel", "bkl"]   # at least 3 different classes, as required

fig, axes = plt.subplots(len(FIGURE_CLASSES), 6, figsize=(18, 3 * len(FIGURE_CLASSES)))
col_titles = ["Original", "Sobel", "Prewitt", "Laplacian", "LoG", "Canny"]

for row, cls in enumerate(FIGURE_CLASSES):
    img = sample_imgs[cls]
    gray = to_gray(img)
    resp = run_detectors(gray)
    axes[row, 0].imshow(img); axes[row, 0].set_ylabel(cls, fontsize=12)
    for col, name in enumerate(DETECTOR_ORDER, start=1):
        axes[row, col].imshow(to_display(resp[name]), cmap="gray")
    for col in range(6):
        axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
        if row == 0:
            axes[row, col].set_title(col_titles[col], fontsize=12)

plt.tight_layout()
plt.savefig(f"{OUT}/figures/task1_edge_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


## 4. Task 2 -- Effect of Noise on Edge Detection


In [ ]:
def add_gaussian_noise(img_rgb, sigma=25):
    noise = np.random.normal(0, sigma, img_rgb.shape).astype(np.float32)
    return np.clip(img_rgb.astype(np.float32) + noise, 0, 255).astype(np.uint8)

def add_salt_pepper_noise(img_rgb, amount=0.03):
    noisy = img_rgb.copy()
    h, w = img_rgb.shape[:2]
    n = int(amount * h * w)
    ys, xs = np.random.randint(0, h, n), np.random.randint(0, w, n)
    half = n // 2
    noisy[ys[:half], xs[:half]] = 255
    noisy[ys[half:], xs[half:]] = 0
    return noisy


def edge_quality(binary_edge, target_density=0.03, density_tolerance=0.03):
    total = binary_edge.size
    edge_px = int((binary_edge > 0).sum())
    if edge_px == 0:
        return 0.0
    density = edge_px / total
    density_score = np.exp(-((density - target_density) ** 2) / (2 * density_tolerance ** 2))
    n_labels, _, stats, _ = cv2.connectedComponentsWithStats((binary_edge > 0).astype(np.uint8), connectivity=8)
    continuity = (stats[1:, cv2.CC_STAT_AREA].max() / edge_px) if n_labels > 1 else 0.0
    return float(100 * density_score * continuity)

def noise_sensitivity(clean_binary, other_binary):
    a, b = (clean_binary > 0), (other_binary > 0)
    denom = a.sum() + b.sum()
    dice = (2 * np.logical_and(a, b).sum() / denom) if denom > 0 else 1.0
    return float(100 * (1 - dice))

def observation(quality, sensitivity):
    q_word = "matches the clean-image edge map almost exactly" if sensitivity < 15 else \
             "moderately diverges from the clean-image edge map" if sensitivity < 60 else \
             "diverges sharply from the clean-image edge map"
    f_word = "a well-connected edge map" if quality > 30 else \
             "a moderately fragmented edge map" if quality > 5 else \
             "a sparse or heavily fragmented edge map"
    return f"{q_word}, with {f_word}."


In [ ]:
# Table 1 rows, following the brief's template exactly. N_SAMPLES representative images
# (one per HAM10000 class) are averaged for each row.
np.random.seed(SEED)

table1_spec = [
    ("Sobel",     "Original", "None",     "None"),
    ("Sobel",     "Noisy",    "Gaussian", "None"),
    ("Sobel",     "Noisy",    "Salt & Pepper", "None"),
    ("Sobel",     "Noisy",    "Gaussian", "Gaussian Filter"),
    ("Sobel",     "Noisy",    "Salt & Pepper", "Median Filter"),
    ("Prewitt",   "Original", "None",     "None"),
    ("Laplacian", "Original", "None",     "None"),
    ("LoG",       "Noisy",    "Gaussian", "Gaussian Filter"),
    ("Canny",     "Original", "None",     "Built-in smoothing"),
    ("Canny",     "Noisy",    "Gaussian", "Gaussian Filter"),
    ("Canny",     "Noisy",    "Salt & Pepper", "Median Filter"),
]

DETECTOR_FN = {
    "Sobel": sobel_magnitude, "Prewitt": prewitt, "Laplacian": laplacian,
    "LoG": log_edge, "Canny": canny_edge,
}

def prepare_input(gray_clean, img_rgb_clean, noise_type, preprocessing):
    '''Returns the grayscale image this row's detector should actually run on.'''
    if noise_type == "None":
        return gray_clean
    noisy_rgb = add_gaussian_noise(img_rgb_clean) if noise_type == "Gaussian" else add_salt_pepper_noise(img_rgb_clean)
    if preprocessing == "Gaussian Filter":
        noisy_rgb = cv2.GaussianBlur(noisy_rgb, (5, 5), 0)
    elif preprocessing == "Median Filter":
        noisy_rgb = cv2.medianBlur(noisy_rgb, 5)
    return to_gray(noisy_rgb)

table1_rows = []
for detector, input_type, noise_type, preprocessing in table1_spec:
    quals, senss = [], []
    for cls, img in sample_imgs.items():
        gray_clean = to_gray(img)
        clean_bin = to_binary(DETECTOR_FN[detector](gray_clean))
        gray_row = prepare_input(gray_clean, img, noise_type, preprocessing)
        row_bin = to_binary(DETECTOR_FN[detector](gray_row))
        quals.append(edge_quality(row_bin))
        senss.append(noise_sensitivity(clean_bin, row_bin) if input_type == "Noisy" else 0.0)
    q, s = float(np.mean(quals)), float(np.mean(senss))
    table1_rows.append({
        "Edge Detector": detector, "Input Image": input_type, "Noise Type": noise_type,
        "Preprocessing": preprocessing, "Edge Quality (%)": round(q, 2),
        "Noise Sensitivity (%)": round(s, 2), "Observations": observation(q, s),
    })

table1 = pd.DataFrame(table1_rows)
table1.to_csv(f"{OUT}/table1.csv", index=False)
table1


## 5. Task 3 -- Canny Parameter Analysis


In [ ]:
canny_configs = [
    ("Canny-1", 30, 100, 3),
    ("Canny-2", 50, 150, 3),
    ("Canny-3", 100, 200, 3),
    ("Canny-4", 50, 150, 5),   # same thresholds as Canny-2, larger smoothing kernel
]

table2_rows = []
for cfg_name, low, high, k in canny_configs:
    quals, counts_px = [], []
    for cls, img in sample_imgs.items():
        gray = to_gray(img)
        edges = canny_edge(gray, low, high, k)
        binary = (edges > 0).astype(np.uint8) * 255
        quals.append(edge_quality(binary))
        counts_px.append(int((binary > 0).sum()))
    q, n_edges = float(np.mean(quals)), float(np.mean(counts_px))
    density_pct = 100 * n_edges / (IMG_SIZE * IMG_SIZE)
    obs = (f"moderate edge density ({density_pct:.1f}% of pixels), a reasonable balance." if 1 < density_pct < 6 else
           f"sparse edge map ({density_pct:.1f}% of pixels), may miss real lesion-boundary detail." if density_pct <= 1 else
           f"dense edge map ({density_pct:.1f}% of pixels), likely includes texture noise as false edges.")
    table2_rows.append({
        "Configuration": cfg_name, "Low Threshold": low, "High Threshold": high,
        "Kernel Size": f"{k}x{k}", "Edge Quality (%)": round(q, 2),
        "Number of Detected Edges": round(n_edges), "Observation": obs,
    })

table2 = pd.DataFrame(table2_rows)
table2.to_csv(f"{OUT}/table2.csv", index=False)

best_cfg = table2.loc[table2["Edge Quality (%)"].idxmax()]
BEST_CANNY = dict(low=int(best_cfg["Low Threshold"]), high=int(best_cfg["High Threshold"]),
                   k=int(best_cfg["Kernel Size"].split("x")[0]))
print(f"Best Canny configuration by Edge Quality: {best_cfg['Configuration']}  "
      f"(low={BEST_CANNY['low']}, high={BEST_CANNY['high']}, k={BEST_CANNY['k']})")
table2


## 6. Task 4 -- Dataset Preparation for Classification


In [ ]:
CONDITIONS = ["Raw", "Filtered", "Edge"]

def apply_condition(img_rgb, condition):
    '''img_rgb: uint8 HxWx3. Returns uint8 HxWx3.'''
    if condition == "Raw":
        return img_rgb
    if condition == "Filtered":
        return cv2.GaussianBlur(img_rgb, (5, 5), sigmaX=1.5)
    if condition == "Edge":
        gray = to_gray(img_rgb)
        edges = canny_edge(gray, **BEST_CANNY)
        return cv2.cvtColor(edges, cv2.COLOR_GRAY2RGB)
    raise ValueError(condition)


class ConditionTransform:
    '''PIL -> PIL. Applied after resize, same as Lab 02's FilterTransform.'''
    def __init__(self, condition): self.condition = condition
    def __call__(self, img):
        return Image.fromarray(apply_condition(np.asarray(img), self.condition))


MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def build_transforms(condition):
    common = [transforms.Resize((IMG_SIZE, IMG_SIZE)), ConditionTransform(condition)]
    train_tf = transforms.Compose(common + [
        transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
        transforms.RandomRotation(20), transforms.ColorJitter(0.15, 0.15, 0.15),
        transforms.ToTensor(), transforms.Normalize(MEAN, STD),
    ])
    eval_tf = transforms.Compose(common + [transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
    return train_tf, eval_tf


class LesionDataset(Dataset):
    def __init__(self, df, tf):
        self.paths, self.labels, self.tf = df["path"].tolist(), df["label"].tolist(), tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        return self.tf(Image.open(self.paths[i]).convert("RGB")), self.labels[i]

def build_loaders(condition, batch_size=BATCH_SIZE):
    train_tf, eval_tf = build_transforms(condition)
    mk = lambda df, tf, sh: DataLoader(LesionDataset(df, tf), batch_size=batch_size, shuffle=sh,
                                       num_workers=NUM_WORKERS, pin_memory=True)
    return (mk(splits["train"], train_tf, True),
            mk(splits["val"],   eval_tf,  False),
            mk(splits["test"],  eval_tf,  False))


# quick sanity check: one example grid, Raw / Filtered / Edge, one image per class
fig, axes = plt.subplots(len(sample_imgs), 3, figsize=(9, 3 * len(sample_imgs)))
for row, (cls, img) in enumerate(sample_imgs.items()):
    for col, cond in enumerate(CONDITIONS):
        axes[row, col].imshow(apply_condition(img, cond), cmap="gray" if cond == "Edge" else None)
        axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
        if row == 0: axes[row, col].set_title(cond)
        if col == 0: axes[row, col].set_ylabel(cls)
plt.tight_layout()
plt.savefig(f"{OUT}/figures/task4_condition_examples.png", dpi=150, bbox_inches="tight")
plt.show()


## 7. Model building


In [ ]:
def build_model(name, n=NUM_CLASSES):
    w = "IMAGENET1K_V1"
    if name == "ResNet50":       m = models.resnet50(weights=w);     m.fc = nn.Linear(m.fc.in_features, n)
    elif name == "ResNet101":    m = models.resnet101(weights=w);    m.fc = nn.Linear(m.fc.in_features, n)
    elif name == "DenseNet121":  m = models.densenet121(weights=w);  m.classifier = nn.Linear(m.classifier.in_features, n)
    else: raise ValueError(name)
    return m.to(DEVICE)

def strip_head(model, name):
    '''Returns (feature_extractor_callable, feature_dim) for a fine-tuned model, used by
    the classical classifiers (Task 5: SVM / Random Forest / KNN).'''
    model.eval()
    if name in ("ResNet50", "ResNet101"):
        feat_dim = model.fc.in_features
        backbone = nn.Sequential(*list(model.children())[:-1])  # drop the fc layer
        extractor = lambda x: backbone(x).flatten(1)
    elif name == "DenseNet121":
        feat_dim = model.classifier.in_features
        features = model.features
        def extractor(x):
            f = features(x)
            f = nn.functional.relu(f, inplace=False)
            f = nn.functional.adaptive_avg_pool2d(f, 1)
            return f.flatten(1)
    else:
        raise ValueError(name)
    return extractor, feat_dim

for name in CNN_MODELS:      # warm the weight cache once
    build_model(name); torch.cuda.empty_cache()
print("weights cached")


## 8. Training, evaluation and feature-extraction functions


In [ ]:
def run_epoch(model, loader, criterion, optimizer=None, scaler=None):
    train = optimizer is not None
    model.train(train)
    tot_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast("cuda", enabled=(DEVICE == "cuda")):
                out = model(x); loss = criterion(out, y)
            if train:
                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            tot_loss += loss.item() * y.size(0)
            correct += (out.argmax(1) == y).sum().item(); n += y.size(0)
    return tot_loss / n, correct / n


@torch.no_grad()
def predict(model, loader, time_inference=False):
    model.eval(); ys, probs = [], []
    t0 = time.time(); n_seen = 0
    for x, y in loader:
        with torch.autocast("cuda", enabled=(DEVICE == "cuda")):
            out = model(x.to(DEVICE, non_blocking=True))
        probs.append(torch.softmax(out.float(), 1).cpu().numpy()); ys.append(y.numpy())
        n_seen += y.size(0)
    y_true, y_prob = np.concatenate(ys), np.concatenate(probs)
    ms_per_image = 1000 * (time.time() - t0) / max(n_seen, 1)
    if time_inference:
        return y_true, y_prob.argmax(1), y_prob, ms_per_image
    return y_true, y_prob.argmax(1), y_prob


def metrics(y_true, y_pred, y_prob):
    return {
        "Accuracy":          accuracy_score(y_true, y_pred) * 100,
        "Precision":         precision_score(y_true, y_pred, average="macro", zero_division=0) * 100,
        "Recall":            recall_score(y_true, y_pred, average="macro", zero_division=0) * 100,
        "F1-score":          f1_score(y_true, y_pred, average="weighted", zero_division=0) * 100,
        "Macro-F1":          f1_score(y_true, y_pred, average="macro", zero_division=0) * 100,
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred) * 100,
        "AUC":               roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro") * 100,
    }


def train_cnn(model_name, condition):
    '''Fine-tunes model_name on this condition. Returns the trained model plus test-set
    metrics/timing, for Table 3's CNN rows and as the feature extractor for Task 5's
    classical classifiers on the same condition.'''
    train_loader, val_loader, test_loader = build_loaders(condition)
    model = build_model(model_name)
    criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    scaler = torch.amp.GradScaler("cuda", enabled=(DEVICE == "cuda"))

    best_acc, best_state = -1, None
    t0 = time.time()
    for ep in range(EPOCHS):
        tl, ta = run_epoch(model, train_loader, criterion, optimizer, scaler)
        vl, va = run_epoch(model, val_loader, criterion)
        scheduler.step()
        if va > best_acc: best_acc, best_state = va, copy.deepcopy(model.state_dict())
        print(f"    [{model_name}/{condition}] ep{ep+1:02d} loss {tl:.3f}/{vl:.3f}  acc {ta*100:.1f}/{va*100:.1f}")
    train_time_s = time.time() - t0

    model.load_state_dict(best_state)
    y_true, y_pred, y_prob, ms_per_image = predict(model, test_loader, time_inference=True)
    del optimizer, scheduler, scaler; torch.cuda.empty_cache()
    m = metrics(y_true, y_pred, y_prob)
    m["Training Time (s)"] = train_time_s
    m["Inference Time (ms)"] = ms_per_image
    return model, m, (y_true, y_pred)


@torch.no_grad()
def extract_features(model, model_name, loader):
    extractor, _ = strip_head(model, model_name)
    feats, ys = [], []
    for x, y in loader:
        with torch.autocast("cuda", enabled=(DEVICE == "cuda")):
            f = extractor(x.to(DEVICE, non_blocking=True))
        feats.append(f.float().cpu().numpy()); ys.append(y.numpy())
    return np.concatenate(feats), np.concatenate(ys)


In [ ]:
CLASSICAL_MODELS = {
    "SVM":           lambda: SVC(kernel="rbf", probability=True, class_weight="balanced", random_state=SEED),
    "Random Forest": lambda: RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=SEED, n_jobs=-1),
    "KNN":           lambda: KNeighborsClassifier(n_neighbors=7),
}

def train_classical(clf_name, X_train, y_train, X_test, y_test):
    clf = CLASSICAL_MODELS[clf_name]()
    t0 = time.time()
    clf.fit(X_train, y_train)
    train_time_s = time.time() - t0

    t0 = time.time()
    y_prob = clf.predict_proba(X_test)
    ms_per_image = 1000 * (time.time() - t0) / max(len(X_test), 1)
    y_pred = y_prob.argmax(1)

    m = metrics(y_test, y_pred, y_prob)
    m["Training Time (s)"] = train_time_s
    m["Inference Time (ms)"] = ms_per_image
    return m, (y_test, y_pred)


## 9. Task 5 -- Run all experiments


In [ ]:
RESULTS_CSV = f"{OUT}/table3_full_long_format.csv"
experiment_results = pd.read_csv(RESULTS_CSV) if os.path.exists(RESULTS_CSV) else pd.DataFrame()
done = set(zip(experiment_results.get("Model", []), experiment_results.get("Condition", [])))

cms = {}

for condition in CONDITIONS:
    print(f"\n{'='*20} condition: {condition} {'='*20}")

    resnet50_model = None
    for model_name in CNN_MODELS:
        key = (model_name, condition)
        if key in done:
            print(f"skip {key} (already in {RESULTS_CSV})")
            if model_name == "ResNet50":
                # need it for the classical classifiers below even if this row was already done
                resnet50_model, _, _ = train_cnn("ResNet50", condition)  # cheap: re-evaluates, doesn't retrain past a completed run in a smarter version
            continue
        print(f"\n-- fine-tuning {model_name} on {condition} --")
        model, m, (y_true, y_pred) = train_cnn(model_name, condition)
        row = {"Model": model_name, "Condition": condition, **m}
        experiment_results = pd.concat([experiment_results, pd.DataFrame([row])], ignore_index=True)
        experiment_results.to_csv(RESULTS_CSV, index=False)
        cms[key] = confusion_matrix(y_true, y_pred)
        np.save(f"{OUT}/confusion/{model_name}_{condition}.npy", cms[key])
        print(f"   -> acc {m['Accuracy']:.2f}  F1 {m['F1-score']:.2f}  ({m['Training Time (s)']/60:.1f} min)")
        if model_name == "ResNet50":
            resnet50_model = model

    # classical classifiers on this condition's fine-tuned ResNet50 features
    train_loader, val_loader, test_loader = build_loaders(condition, batch_size=64)
    X_train, y_train = extract_features(resnet50_model, "ResNet50", train_loader)
    X_test,  y_test  = extract_features(resnet50_model, "ResNet50", test_loader)

    for clf_name in CLASSICAL_MODELS:
        key = (clf_name, condition)
        if key in done:
            print(f"skip {key} (already in {RESULTS_CSV})"); continue
        print(f"\n-- {clf_name} on {condition} (ResNet50 deep features) --")
        m, (yt, yp) = train_classical(clf_name, X_train, y_train, X_test, y_test)
        row = {"Model": clf_name, "Condition": condition, **m}
        experiment_results = pd.concat([experiment_results, pd.DataFrame([row])], ignore_index=True)
        experiment_results.to_csv(RESULTS_CSV, index=False)
        cms[key] = confusion_matrix(yt, yp)
        np.save(f"{OUT}/confusion/{clf_name}_{condition}.npy", cms[key])
        print(f"   -> acc {m['Accuracy']:.2f}  F1 {m['F1-score']:.2f}")

    del resnet50_model; torch.cuda.empty_cache()

experiment_results.to_csv(RESULTS_CSV, index=False)
print(f"\nAll done. {len(experiment_results)} rows saved to {RESULTS_CSV}")


## 10. Table 3 -- Cross-Lab Classification Performance Comparison


In [ ]:
experiment_results = pd.read_csv(RESULTS_CSV)
ROW_ORDER = ["SVM", "Random Forest", "KNN", "CNN Model 1", "CNN Model 2"]
name_map = {"ResNet50": "CNN Model 1 (ResNet50)", "DenseNet121": "CNN Model 2 (DenseNet121)"}

acc_pivot = experiment_results.pivot(index="Model", columns="Condition", values="Accuracy")
acc_pivot = acc_pivot.rename(columns={"Raw": "Accuracy Raw (Lab 1) (%)",
                                       "Filtered": "Accuracy Filtered (Lab 2) (%)",
                                       "Edge": "Accuracy Edge (Lab 3) (%)"})

# Precision/Recall/F1/timing shown for the Edge condition -- Lab 3's own new contribution;
# the full 15-row table with every metric under every condition is table3_full_long_format.csv
edge_extra = experiment_results[experiment_results["Condition"] == "Edge"].set_index("Model")[
    ["Precision", "Recall", "F1-score", "Training Time (s)", "Inference Time (ms)"]]

table3 = acc_pivot.join(edge_extra)
table3.index = [name_map.get(i, i) for i in table3.index]
table3 = table3.reindex([name_map.get(r, r) for r in ROW_ORDER])
table3 = table3.round(2)
table3


## 11. Task 6 -- Confusion Matrices and Metric Comparison


In [ ]:
# best-performing row overall, by mean accuracy across the three conditions
mean_acc = experiment_results.groupby("Model")["Accuracy"].mean().sort_values(ascending=False)
best_row = mean_acc.index[0]
print("Best-performing row overall (mean accuracy across conditions):", best_row, f"({mean_acc.iloc[0]:.2f}%)")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, condition in zip(axes, CONDITIONS):
    cm = cms.get((best_row, condition))
    if cm is None:
        cm = np.load(f"{OUT}/confusion/{best_row}_{condition}.npy")
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES,
                ax=ax, cbar=False)
    ax.set_title(f"{best_row} -- {condition}"); ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout()
plt.savefig(f"{OUT}/figures/task6_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
metric_cols = ["Accuracy", "Precision", "Recall", "F1-score"]
plot_df = experiment_results[experiment_results["Model"] == best_row].set_index("Condition")[metric_cols].reindex(CONDITIONS)

ax = plot_df.plot(kind="bar", figsize=(9, 5))
ax.set_ylabel("%"); ax.set_title(f"{best_row}: Accuracy / Precision / Recall / F1 by condition")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(f"{OUT}/figures/task6_metric_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

plot_df
